# putEMG — Full-Dataset Retraining

Loads the best model weights from `3_model_evaluation.ipynb` and retrains
that model on the **entire** dataset (no held-out test set).

Use this after evaluation to produce the final deployment weights.

**Workflow:**
1. Load checkpoint from `weights/` (produced by `3_model_evaluation.ipynb`)
2. Load all data — no train/test split
3. Retrain with early stopping on training loss + ReduceLROnPlateau
4. Save final weights to `weights/`

In [1]:
import os
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import models
from data_utils import train, evaluate, full_loader

num_classes 7
num_training_ex 200
X: (1400, 1, 24, 1500), Y: (1400,)
Train: 1008 | Dev: 112 | Test: 280


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---
## Configuration

Set the checkpoint path and training parameters here.

In [3]:
# ── Weights ───────────────────────────────────────────────────────────────────
WEIGHTS_DIR = 'weights'
# Path to the checkpoint produced by 3_model_evaluation.ipynb.
# Change the filename to retrain a different model.
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'EMG_TCN_best.pt')   # edit as needed

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE = 16
MAX_EPOCHS = 80         # upper limit; early stopping usually kicks in sooner
PATIENCE   = 15         # early-stopping patience on training loss
MIN_DELTA  = 0.002      # minimum improvement to reset patience
LR         = 1e-3       # initial learning rate

---
## Load Checkpoint

Reconstructs the model architecture and loads the pre-trained weights
from `3_model_evaluation.ipynb`.

In [4]:
# Model name → constructor map (lambdas accept dropout from the checkpoint)
model_registry = {
    'EEGNet':         lambda d: models.EEGNet(dropout_rate=d),
    'ShallowConvNet': lambda d: models.ShallowConvNet(dropout_rate=d),
    'DeepConvNet':    lambda d: models.DeepConvNet(dropout_rate=d),
    'CNN_LSTM':       lambda d: models.CNN_LSTM(dropout_rate=d),
    'EMG_TCN':        lambda d: models.EMG_TCN(dropout_rate=d),
}

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

model_name = checkpoint['model_name']
dropout    = checkpoint['dropout']
eval_acc   = checkpoint['test_acc']

print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Model        : {model_name}")
print(f"Dropout      : {dropout}")
print(f"Eval acc     : {eval_acc * 100:.2f}%")

model = model_registry[model_name](dropout).to(device)
model.load_state_dict(checkpoint['state_dict'])

print("\nWeights loaded. Starting from evaluation checkpoint.")

Checkpoint   : weights/EMG_TCN_best.pt
Model        : EMG_TCN
Dropout      : 0.25
Eval acc     : 93.57%

Weights loaded. Starting from evaluation checkpoint.


---
## Retrain on Full Dataset

Fine-tunes from the evaluation checkpoint on all available data.
Early stopping monitors training loss (no validation set available).

In [5]:
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                              patience=5, min_lr=1e-6)
criterion = nn.CrossEntropyLoss()

best_loss  = float('inf')
best_state = None
bad_epochs = 0

for epoch in range(MAX_EPOCHS):
    tr_loss = train(model, full_loader, criterion, optimizer, device)
    tr_acc  = evaluate(model, full_loader, device)
    curr_lr = optimizer.param_groups[0]['lr']

    # Best-state checkpoint (tracked by training loss)
    if tr_loss < best_loss:
        best_loss  = tr_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Learning-rate scheduler (monitors training loss)
    scheduler.step(tr_loss)

    # Early stopping
    if tr_loss <= (best_loss + MIN_DELTA):
        bad_epochs = 0
    else:
        bad_epochs += 1

    print(f"  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  acc={tr_acc*100:.2f}%  "
          f"best_loss={best_loss:.4f}  lr={curr_lr:.2e}")

    if bad_epochs >= PATIENCE:
        print(f"  Early stopping at epoch {epoch+1}.")
        break

# Restore best weights
model.load_state_dict(best_state)
print(f"\nRetrain complete. Best training loss: {best_loss:.4f}")

KeyboardInterrupt: 

---
## Save Final Weights

In [ ]:
os.makedirs(WEIGHTS_DIR, exist_ok=True)

save_path = os.path.join(WEIGHTS_DIR, f"{model_name}_final.pt")
torch.save({
    'model_name':  model_name,
    'dropout':     dropout,
    'eval_acc':    eval_acc,    # accuracy from 3_model_evaluation.ipynb
    'state_dict':  best_state,
}, save_path)

print(f"Final weights saved → {save_path}")